# Article Full Text Retrieval Exploration

**Date:** 2026-01-08

**Goal:** Explore methods to retrieve full text articles associated with Dryad/Zenodo data papers from the Fuster et al. annotated dataset.

## Hypotheses to Test

1. **H1: Full text already in Excel** - The `full_text` column might contain article full text (not just abstract)
2. **H2: URL links in Excel** - Columns like `cited_articles` or other fields might contain direct links to full text articles
3. **H3: Dryad/Zenodo API metadata** - Repository APIs might provide article links in metadata (e.g., "related identifiers", "references", "publication")

## Test Strategy

- Select 2-3 examples from Dryad datasets
- Select 2-3 examples from Zenodo datasets
- For each example, test all three hypotheses
- Document successful retrieval methods

In [ ]:
import pandas as pd
import requests
from llm_metadata.dryad import get_dataset
from llm_metadata.zenodo import get_record_by_doi
import json
from pprint import pprint
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
dotenv_path = Path('../.env')
load_dotenv(dotenv_path)

print(f"Environment loaded from: {dotenv_path.absolute()}")
print(f"Zenodo token present: {'Yes' if os.getenv('ZENODO_ACCESS_TOKEN') else 'No'}")
print(f"OpenAI key present: {'Yes' if os.getenv('OPENAI_API_KEY') else 'No'}")

## Load Annotated Dataset

In [ ]:
# Load raw annotated data
df = pd.read_excel('../data/dataset_092624.xlsx')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nSource distribution:")
print(df['source'].value_counts())

## Select Test Samples

Pick valid datasets (valid_yn='yes') from both Dryad and Zenodo

In [ ]:
# Filter valid datasets
valid_df = df[df['valid_yn'] == 'yes'].copy()

# Separate by source
dryad_samples = valid_df[valid_df['source'] == 'dryad'].head(3)
zenodo_samples = valid_df[valid_df['source'] == 'zenodo'].head(3)

print("Dryad samples:")
print(dryad_samples[['id', 'url', 'title', 'source']].to_string())

print("\n" + "="*80 + "\n")

print("Zenodo samples:")
print(zenodo_samples[['id', 'url', 'title', 'source']].to_string())

## Hypothesis 1: Full Text Already in Excel

Check if the `full_text` column contains actual article text or just abstracts

In [ ]:
def analyze_full_text_column(df_sample, source_name):
    """Analyze the full_text column to determine if it's abstract or full article"""
    print(f"\n{'='*80}")
    print(f"{source_name} - Full Text Analysis")
    print(f"{'='*80}\n")
    
    for idx, row in df_sample.iterrows():
        print(f"ID: {row['id']}")
        print(f"URL: {row['url']}")
        print(f"Title: {row['title'][:80]}...")
        
        full_text = str(row['full_text']) if pd.notna(row['full_text']) else ""
        print(f"Full text length: {len(full_text)} characters")
        print(f"Full text preview (first 300 chars):\n{full_text[:300]}...")
        
        # Heuristic: abstracts are typically < 5000 chars, full articles > 10000
        if len(full_text) > 10000:
            print("⚠️ LIKELY FULL ARTICLE TEXT")
        elif len(full_text) > 2000:
            print("❓ POSSIBLY EXTENDED ABSTRACT OR DESCRIPTION")
        else:
            print("📄 LIKELY ABSTRACT ONLY")
        
        print("\n" + "-"*80 + "\n")

# Test Hypothesis 1 for both sources
analyze_full_text_column(dryad_samples, "DRYAD")
analyze_full_text_column(zenodo_samples, "ZENODO")

## Hypothesis 2: URL Links in Excel

Search for article URLs in various columns (cited_articles, url.1, etc.)

In [ ]:
def search_article_links_in_excel(df_sample, source_name):
    """Search for article links in Excel columns"""
    print(f"\n{'='*80}")
    print(f"{source_name} - Article Links in Excel")
    print(f"{'='*80}\n")
    
    # Columns that might contain article links
    potential_link_columns = [
        'cited_articles', 'url.1', 'geospatial_info_article_text',
        'data_type_article_text', 'species_article_text'
    ]
    
    for idx, row in df_sample.iterrows():
        print(f"ID: {row['id']}")
        print(f"Dataset URL: {row['url']}")
        print(f"Title: {row['title'][:60]}...")
        
        found_links = False
        for col in potential_link_columns:
            if col in df.columns and pd.notna(row[col]):
                value = str(row[col])
                # Check if contains URL patterns
                if 'http' in value.lower() or 'doi' in value.lower() or 'www.' in value:
                    print(f"\n✅ Found potential link in '{col}':")
                    print(f"   {value[:200]}...")
                    found_links = True
        
        if not found_links:
            print("\n❌ No article links found in Excel columns")
        
        print("\n" + "-"*80 + "\n")

# Test Hypothesis 2
search_article_links_in_excel(dryad_samples, "DRYAD")
search_article_links_in_excel(zenodo_samples, "ZENODO")

## Hypothesis 3: Dryad/Zenodo API Metadata

Query repository APIs to find article links in metadata

### Test Dryad API for Article Links

In [ ]:
def extract_dryad_article_info(row):
    """Extract article information from Dryad API"""
    print(f"\n{'='*80}")
    print(f"Dryad Dataset ID: {row['id']}")
    print(f"Dataset URL: {row['url']}")
    print(f"{'='*80}\n")
    
    try:
        # Extract DOI from URL
        doi = row['url'].replace('https://doi.org/', '')
        print(f"Querying Dryad API for DOI: {doi}")
        
        # Get dataset metadata
        dataset = get_dataset(doi)
        
        if dataset:
            print("\n📦 Dataset metadata retrieved")
            
            # Check for related publication/article
            if 'relatedWorks' in dataset:
                print("\n✅ Related Works Found:")
                for work in dataset['relatedWorks']:
                    print(f"\n  Relationship: {work.get('relationship', 'N/A')}")
                    print(f"  Identifier Type: {work.get('identifierType', 'N/A')}")
                    print(f"  Identifier: {work.get('identifier', 'N/A')}")
            
            # Check for publication DOI or URL
            if 'publicationDOI' in dataset:
                print(f"\n✅ Publication DOI: {dataset['publicationDOI']}")
            
            # Check relatedIdentifiers
            if 'relatedIdentifiers' in dataset:
                print("\n✅ Related Identifiers:")
                for identifier in dataset['relatedIdentifiers']:
                    print(f"  {identifier}")
            
            # Print full metadata structure to explore
            print("\n📋 Available metadata keys:")
            print(list(dataset.keys()))
            
            return dataset
        else:
            print("❌ Failed to retrieve dataset metadata")
            return None
            
    except Exception as e:
        print(f"❌ Error querying Dryad API: {e}")
        return None

# Test with Dryad samples
print("\n" + "#"*80)
print("# TESTING DRYAD API FOR ARTICLE LINKS")
print("#"*80)

dryad_metadata_results = []
for idx, row in dryad_samples.iterrows():
    result = extract_dryad_article_info(row)
    dryad_metadata_results.append(result)
    print("\n" + "-"*80)

### Test Zenodo API for Article Links

In [ ]:
def extract_zenodo_article_info(row):
    """Extract article information from Zenodo API"""
    print(f"\n{'='*80}")
    print(f"Zenodo Dataset ID: {row['id']}")
    print(f"Dataset URL: {row['url']}")
    print(f"{'='*80}\n")
    
    try:
        # Extract DOI from URL
        doi = row['url'].replace('https://doi.org/', '')
        print(f"Querying Zenodo API for DOI: {doi}")
        
        # Get record metadata
        record = get_record_by_doi(doi)
        
        if record:
            print("\n📦 Record metadata retrieved")
            
            # Check metadata section
            metadata = record.get('metadata', {})
            
            # Check for related identifiers
            if 'related_identifiers' in metadata:
                print("\n✅ Related Identifiers Found:")
                for rel_id in metadata['related_identifiers']:
                    print(f"\n  Relation: {rel_id.get('relation', 'N/A')}")
                    print(f"  Scheme: {rel_id.get('scheme', 'N/A')}")
                    print(f"  Identifier: {rel_id.get('identifier', 'N/A')}")
            
            # Check for references
            if 'references' in metadata:
                print("\n✅ References Found:")
                for ref in metadata['references']:
                    print(f"  {ref}")
            
            # Check for journal/publication info
            if 'journal' in metadata:
                print(f"\n✅ Journal Info: {metadata['journal']}")
            
            # Check for DOI in alternate identifiers
            if 'alternate_identifiers' in metadata:
                print("\n✅ Alternate Identifiers:")
                for alt_id in metadata['alternate_identifiers']:
                    print(f"  {alt_id}")
            
            # Print available metadata keys
            print("\n📋 Available metadata keys:")
            print(list(metadata.keys()))
            
            return record
        else:
            print("❌ Failed to retrieve record metadata")
            return None
            
    except Exception as e:
        print(f"❌ Error querying Zenodo API: {e}")
        return None

# Test with Zenodo samples
print("\n" + "#"*80)
print("# TESTING ZENODO API FOR ARTICLE LINKS")
print("#"*80)

zenodo_metadata_results = []
for idx, row in zenodo_samples.iterrows():
    result = extract_zenodo_article_info(row)
    zenodo_metadata_results.append(result)
    print("\n" + "-"*80)

## Summary: Article Retrieval Methods

Compile findings from all three hypotheses

In [ ]:
print("\n" + "="*80)
print("SUMMARY OF FINDINGS")
print("="*80 + "\n")

print("✅ = Successful method for article retrieval")
print("❌ = Method did not yield article full text")
print("❓ = Method requires further investigation\n")

print("-"*80)
print("Hypothesis 1: Full text in Excel 'full_text' column")
print("-"*80)
print("Result: [TO BE FILLED BASED ON EXECUTION]\n")

print("-"*80)
print("Hypothesis 2: Article URLs in Excel columns")
print("-"*80)
print("Result: [TO BE FILLED BASED ON EXECUTION]\n")

print("-"*80)
print("Hypothesis 3: Repository API metadata (related_identifiers, etc.)")
print("-"*80)
print("Result: [TO BE FILLED BASED ON EXECUTION]\n")

print("="*80)
print("RECOMMENDED APPROACH")
print("="*80)
print("[TO BE DETERMINED BASED ON RESULTS]")

## Next Steps

Based on successful methods identified:

1. Implement automated article link extraction for all valid datasets
2. Create functions to download/access full text (respecting open access restrictions)
3. Build a small database linking dataset DOIs to article DOIs/URLs
4. Explore article retrieval APIs (Unpaywall, CrossRef, etc.) for DOI-based full text access